# Mini Project: Stroke Dataset Analysis 

**Author:** Yasir Binsaif  

**Purpose:**  

A **stroke** happens when the blood supply to part of the brain is blocked or when a blood vessel bursts, leading to cell death and brain damage. (Adapted from CDC)  

Predicting stroke risk early in people with high likelihood can help with prevention and prompt intervention.

This project aims at exploring patients' health data to identify patterns and answer:  
1. Do patients with high average glucose levels have a higher chance of developing a stroke? 
2. Do male and female patients show different stroke rates?  

Dataset Source: [Kaggle Stroke Prediction Dataset](https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset)



## Importing Libraries and Loading Data 

In [1]:
# Core libraries 
import pandas as pd
import numpy as np

# Visualization libraries 
import matplotlib.pyplot as plt 

# Load dataset 
stroke_df = pd.read_csv("healthcare-dataset-stroke-data.csv")


## Data Inspection 

### Dataset Shape
How many rows and columns are in the dataset? 

In [2]:
stroke_df.shape

(5110, 12)

### Preview of the First 5 Rows 


In [3]:
stroke_df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


The dataset contains the following variable types:

**Numeric variables**
- age (Age of the patient in years)
- avg_glucose_level (Average blood glucose, measured in mg/dl)
- bmi (Body Mass Index)

**Binary variables (0 = No, 1 = Yes)** 
- hypertension (Whether the patient has hypertension)
- heart_disease (Whether the patient has a heart disease)
- stroke (Whether the patient had a stroke) (target variable)

**Categorical variables** 
- gender (Male, Female, Other)  
- ever_married (Yes/No)  
- work_type (Private, Self-employed, Other)  
- Residence_type (Urban, Rural)  
- smoking_status (Formerly smoked, Never smoked, Smoked, Unknown)  

### Data Types and Non-Null Count

In [4]:
stroke_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 entries, 0 to 5109
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 5110 non-null   int64  
 1   gender             5110 non-null   object 
 2   age                5110 non-null   float64
 3   hypertension       5110 non-null   int64  
 4   heart_disease      5110 non-null   int64  
 5   ever_married       5110 non-null   object 
 6   work_type          5110 non-null   object 
 7   Residence_type     5110 non-null   object 
 8   avg_glucose_level  5110 non-null   float64
 9   bmi                4909 non-null   float64
 10  smoking_status     5110 non-null   object 
 11  stroke             5110 non-null   int64  
dtypes: float64(3), int64(4), object(5)
memory usage: 479.2+ KB


### Missing Values
Total missing Values in each column.

In [5]:
stroke_df.isnull().sum()

id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

Percentage of values that are missing from `bmi`

In [6]:
missing_pct = (stroke_df["bmi"].isnull().sum()/len(stroke_df["bmi"])) * 100 
print(f"{round(missing_pct, 2)}%")

3.93%


### Descriptive Statistics 
For columns with numeric values 

In [7]:
pd.set_option("display.precision", 2)
stroke_df[["age","avg_glucose_level", "bmi"]].describe()

,age,avg_glucose_level,bmi
count,5110.00,5110.00,4909.00
mean,43.23,106.15,28.89
std,22.61,45.28,7.85
min,0.08,55.12,10.30
25%,25.00,77.25,23.50
50%,45.00,91.88,28.10
75%,61.00,114.09,33.10
max,82.00,271.74,97.60


For columns with binary values 

In [8]:
total_patients = len(stroke_df)

hypertension_patients_num = len(stroke_df[stroke_df["hypertension"] == 1])
heart_disease_patients_num = len(stroke_df[stroke_df["heart_disease"]== 1])
stroke_patients_num = len(stroke_df[stroke_df["stroke"] == 1])

print(f"Hypertension: {hypertension_patients_num} patients ({(hypertension_patients_num/total_patients) * 100:.2f}%)")
print(f"Heart disease: {heart_disease_patients_num} patients ({(heart_disease_patients_num/total_patients) * 100:.2f}%)")
print(f"Stroke: {stroke_patients_num} patients ({(stroke_patients_num/total_patients) * 100:.2f}%)")

Hypertension: 498 patients (9.75%)
Heart disease: 276 patients (5.40%)
Stroke: 249 patients (4.87%)


For columns with categorical values

In [9]:
num_of_males = len(stroke_df[stroke_df["gender"] == "Male"])
num_of_females = len(stroke_df[stroke_df["gender"] == "Female"])
num_of_other_genders = (~stroke_df["gender"].isin(["Male", "Female"])).sum()

print(f"Males: {num_of_males} patients ({(num_of_males/total_patients) * 100:.2f}%)")
print(f"Females: {num_of_females} patients ({(num_of_females/total_patients) * 100:.2f}%)")
print(f"Other: {num_of_other_genders} patients ({(num_of_other_genders/total_patients) * 100:.2f}%)")

Males: 2115 patients (41.39%)
Females: 2994 patients (58.59%)
Other: 1 patients (0.02%)


Other categorical columns (e.g., ever_married, work_type) are included in the dataset but are not explored further in this analysis.

## Data Cleaning

### Standardize Column Names
- Convert column names to lowercase.
- Remove any leading/trailing whitespace.  
- Fix the inconsistency in `Residence_type` by aligning it with the rest. 

In [10]:
stroke_df.columns = stroke_df.columns.str.strip().str.lower() 

### Handle Missing Values

- Column `bmi` has approximately 3.9% missing values.  

- **Decision 1 (Drop vs Impute):** BMI is an important feature, therefore imputation was chosen.  

- **Decision 2 (Mean vs Median):** BMI distribution is right-skewed, so the median was used instead of the mean.  

- No other columns required action.  

In [11]:
# Check skewness 
print(f"Skewness of BMI column: {stroke_df['bmi'].skew():.2f}") # > 0 = right skewness 

# Missing values before 
print(f"BMI missing values before: {stroke_df["bmi"].isnull().sum()}")

# Replace missing values with median
stroke_df["bmi"] = stroke_df["bmi"].fillna(stroke_df["bmi"].median())

# Missing values after 
print(f"BMI missing values after: {stroke_df["bmi"].isnull().sum()}")

Skewness of BMI column: 1.06
BMI missing values before: 201
BMI missing values after: 0


### Check for Duplicates

- Checked for duplicates with `.duplicated().sum()`
- **Decision:** No duplicates were found, so no action was required. 

In [12]:
# Checking duplicates and printing the value
print(stroke_df.duplicated().sum())

0


### Remove Irrelevant Column
- Column `ID` is only a unique identifier, it adds no analytical value, therefore it will be dropped. 

In [13]:
stroke_df = stroke_df.drop(columns=["id"])

### Validate Data Types 

- **Numeric columns** (age, avg_glucose_level, bmi) are already numeric. 
- **Categorical columns** (gender, ever_married, work_type, residence_type, smoking_status) need to be converted to `category`.  
- **Binary columns** (hypertension, heart_disease, stroke) will be treated as numeric for this analysis.  


In [14]:
# Check current data types 
print("\nBefore conversion:")
print(stroke_df.dtypes)

# Convert categorical columns to category dtype
categorical_cols = ["gender", "ever_married", "work_type", "residence_type", "smoking_status"]
stroke_df[categorical_cols] = stroke_df[categorical_cols].astype("category")

# Verify conversion 
print("\nAfter conversion:")
print(stroke_df[categorical_cols].dtypes)


Before conversion:
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object

After conversion:
gender            category
ever_married      category
work_type         category
residence_type    category
smoking_status    category
dtype: object


### Check for Outliers and Inconsistent Values
- Set broad ranges for numeric variables (e.g., age, bmi, avg_glucose_level) to remove impossible or biologically implausible values. 
- Retain extreme but plausible cases for later outlier analysis.  
- Categorical values are consistent; no action required.

**Reasoning**
- **Age:** 2-120 years. Stroke in infants under 2 is extremely rare; values below 2 were removed. The upper limit is set at 120 years.  

- **Average glucose level:** 20-600 mg/dl. Values outside this range are physiologically implausible and likely data entry error. Extreme values within the range will be reviewed in the outlier analysis.  

- **BMI:** 10-100. Values outside this range are almost always invalid. Extreme but plausible BMIs (e.g, > 80) are retained for outlier inspection. 


Set boundaries 

In [15]:
# Length before cleaning 
n_rows_before_cleaning = len(stroke_df)

# Set boundaries 
stroke_df = stroke_df[
    (stroke_df["age"].between(2,120)) &
    (stroke_df["avg_glucose_level"].between(20, 600)) &
    (stroke_df["bmi"].between(10,100))
    ]

# Length after cleaning 
n_rows_after_cleaning = len(stroke_df)

print(f"Rows before cleaning: {n_rows_before_cleaning} | After cleaning: {n_rows_after_cleaning} | Removed: {n_rows_before_cleaning - n_rows_after_cleaning}")


Rows before cleaning: 5110 | After cleaning: 4990 | Removed: 120


Outlier analysis 